# Day 9 | Lab 9.1: LangSmith Instrumentation + Custom Callbacks

**Duration:** ~1.75 hours

**Scenario.** *Customer Support Router Agentic RAG System* — preserved from the AGN source notebook. An AI products & hardware company routes inbound customer queries through a LangGraph agent that categorises (Technical / Billing / General), reads sentiment, retrieves from a Chroma knowledge base, and either generates a grounded reply or escalates to a human. We layer **production observability** on top: LangSmith auto-tracing, a custom `BaseCallbackHandler` that writes a structured **JSON-lines audit log** for compliance, LangSmith **datasets + evaluators** for regression testing, and the **LangSmith Hub** for prompt versioning across CI/CD.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Configure LangSmith in any LangChain / LangGraph project with `LANGSMITH_TRACING=true` + a project name.
2. Subclass `BaseCallbackHandler` and emit structured JSON-lines audit events that survive a regulator review.
3. Create a LangSmith **dataset**, attach a **correctness evaluator**, and run an **experiment** as a regression check.
4. Push and pull versioned prompts via **LangSmith Hub** so prompt changes flow through your CI/CD pipeline.
5. Reason about when to send traces to LangSmith Cloud vs keep them on-prem (regulated-data scenarios).

**Tools.** LangChain v1 · LangGraph v1 · `langsmith` · `langchain-openai` · `langchain-chroma` · OpenAI `gpt-5-mini` (preserved from source) · text-embedding-3-small.

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langgraph>=1.0'
# !pip install -q langchain-chroma langchain-community 'langsmith>=0.4' python-dotenv


Pinned versions above match the curriculum stack (CLAUDE.md §2.1). The original source pinned `langchain==0.3.14` — superseded by LangChain v1 (Nov 2025), which is fully backward-compatible with the v0.3 tracing semantics that LangSmith depends on.

## 2. API Key Configuration

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely
# on environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY', 'LANGSMITH_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


In [ ]:
# LangSmith picks up these env vars automatically — every LangChain / LangGraph call from
# this point on becomes a tracked run in the dashboard.
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_PROJECT'] = 'eclerx-day9-lab9.1-customer-support-router'
# Optional — pin a specific endpoint (US default; switch to .eu for EU-residency tenants)
os.environ.setdefault('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')

print('LangSmith tracing →', os.environ['LANGSMITH_TRACING'])
print('LangSmith project →', os.environ['LANGSMITH_PROJECT'])


## 3. Business Scenario

An AI products & hardware company answers thousands of inbound support queries every day. The team has built a LangGraph **router agent** with four reply paths — *Technical* (RAG over product docs), *Billing* (RAG over policies), *General* (RAG over FAQ), and *Human Escalation* (negative sentiment).

It works. But the **head of risk** has three questions before this can go live in production:

1. **Can we replay any past customer interaction?** *(LangSmith auto-tracing — §6.)*
2. **Do we have a tamper-evident audit log of every LLM call, tool call, and error?** *(Custom callback — §7.)*
3. **If somebody changes a routing prompt, will we catch the regression before it ships?** *(Datasets + evaluators + Hub — §8 & §9.)*

This lab adds the observability + governance layer that answers all three.

## 4. Load Company Knowledge Base

We embed a small knowledge base inline so the lab runs offline. The source notebook downloaded a JSON from Google Drive via `gdown`; we replace that with a synthetic 12-document KB covering technical, billing, and general categories.

In [ ]:
# Inline synthetic knowledge base — replaces the source's gdown download for offline reproducibility.
import json
from pathlib import Path

knowledge_base = [
    {'text': 'Our pre-trained models include vision (CLIP-style), speech (Whisper-style), and text (Llama-3 fine-tunes). They ship with example notebooks.', 'metadata': {'category': 'technical'}},
    {'text': 'On-prem deployment is supported via the AcmeAI Edge appliance — Kubernetes-based, runs Llama 3 70B on 2x H100.', 'metadata': {'category': 'technical'}},
    {'text': 'Hardware troubleshooting: if the GPU light blinks red, run acmectl diagnose --gpu; common cause is loose NVLink bridge.', 'metadata': {'category': 'technical'}},
    {'text': 'AcmeAI SDK supports Python 3.10+, Node 20+, and Java 17. The REST API is OpenAPI 3.1 compliant.', 'metadata': {'category': 'technical'}},
    {'text': 'We accept Visa, Mastercard, Amex, ACH bank transfer, and wire. Crypto is not supported.', 'metadata': {'category': 'billing'}},
    {'text': 'Invoices are emailed on the 1st of each month. To download past invoices, log in and visit Account → Billing → Invoices.', 'metadata': {'category': 'billing'}},
    {'text': 'You can update your billing info under Account → Billing → Payment Methods. Changes take effect immediately.', 'metadata': {'category': 'billing'}},
    {'text': 'Refunds are processed within 7 business days. We refund pro-rata on cancellation, no questions asked, within 30 days of purchase.', 'metadata': {'category': 'billing'}},
    {'text': 'Our refund policy: full refund within 30 days, pro-rata thereafter. Contact billing@acmeai.example.', 'metadata': {'category': 'general'}},
    {'text': 'Standard shipping is 3–5 business days within the US. International shipping is 7–14 business days; duties not included.', 'metadata': {'category': 'general'}},
    {'text': 'Working hours: Mon–Fri 8am–8pm Eastern. Weekend support is available for Enterprise customers only.', 'metadata': {'category': 'general'}},
    {'text': 'You can reach support at support@acmeai.example or +1-555-ACME-HELP. Average first response: under 4 hours.', 'metadata': {'category': 'general'}},
]

Path('./router_agent_documents.json').write_text(json.dumps(knowledge_base, indent=2))
print(f'Wrote {len(knowledge_base)} documents to ./router_agent_documents.json')
knowledge_base[:3]


In [6]:
import json

with open("./router_agent_documents.json", "r") as f:
    knowledge_base = json.load(f)

knowledge_base[:3]

[{'text': 'Question: How do I integrate your AI product with my existing CRM system? Answer: You can integrate our AI product with your CRM using our API. Refer to the API documentation available on our website for step-by-step guidance.',
  'metadata': {'category': 'technical'}},
 {'text': 'Question: What programming languages are supported by your SDK? Answer: Our SDK supports Python, Java, and JavaScript. Additional language support is planned for future updates.',
  'metadata': {'category': 'technical'}},
 {'text': 'Question: Can your AI models run on-premise? Answer: Yes, our AI models can be deployed on-premise. We provide deployment guides for various environments.',
  'metadata': {'category': 'technical'}}]

In [ ]:
from langchain_core.documents import Document
from tqdm import tqdm

processed_docs: list[Document] = []
for doc in tqdm(knowledge_base):
    processed_docs.append(Document(page_content=doc['text'], metadata=doc['metadata']))

processed_docs[:3]


## 5. Build the LangGraph Router Agent

Build the Chroma vector store, define the `CustomerSupportState`, the four router nodes (`categorize_inquiry`, `analyze_inquiry_sentiment`, `generate_*_response`, `escalate_to_human_agent`), and compile the LangGraph workflow. Cells below preserve the source logic — every LLM call, retrieval, and graph edge is **already** auto-traced to LangSmith because of the env vars set in §2.

In [8]:
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
# Clean any prior persisted Chroma collection so reruns do not collide
import shutil, os
if os.path.exists('./knowledge_base'):
    shutil.rmtree('./knowledge_base')


In [10]:
from langchain_chroma import Chroma

kbase_db = Chroma.from_documents(documents=processed_docs,
                                  collection_name='knowledge_base',
                                  embedding=openai_embed_model,
                                  # need to set the distance function to cosine else it uses euclidean by default
                                  # check https://docs.trychroma.com/guides#changing-the-distance-function
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory="./knowledge_base")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [11]:
kbase_search = kbase_db.as_retriever(search_type="similarity_score_threshold",
                                     search_kwargs={"k": 3, "score_threshold": 0.2})

### 5.1 Customer Inquiry State

We create a `CustomerSupportState` typed dictionary to keep track of each interaction:
- **customer_query**: The text of the customer's question
- **query_category**: Technical, Billing, or General (used for routing)
- **query_sentiment**: Positive, Neutral, or Negative (used for routing)
- **final_response**: The system's response to the customer


In [12]:
from typing import TypedDict, Literal
from pydantic import BaseModel

class CustomerSupportState(TypedDict):
    customer_query: str
    query_category: str
    query_sentiment: str
    final_response: str

class QueryCategory(BaseModel):
    categorized_topic: Literal['Technical', 'Billing', 'General']

class QuerySentiment(BaseModel):
    sentiment: Literal['Positive', 'Neutral', 'Negative']

### 5.2 Node Functions

Each function below represents a stage in processing a customer inquiry:
1. **categorize_inquiry** — classifies the query into Technical / Billing / General.
2. **analyze_inquiry_sentiment** — Positive / Neutral / Negative.
3. **generate_*_response** — RAG over the Chroma KB, scoped by metadata filter.
4. **escalate_to_human_agent** — fallback for negative sentiment.
5. **determine_route** — conditional edge function that picks the next node.


In [ ]:
from langchain_openai import ChatOpenAI

# Source pinned gpt-5-mini; preserved here. Swap to gpt-4.1-mini if you do not have access.
llm = ChatOpenAI(model='gpt-5-mini')


In [14]:
def categorize_inquiry(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Classify the customer query into Technical, Billing, or General.
    """

    query = support_state["customer_query"]
    ROUTE_CATEGORY_PROMPT = """Act as a customer support agent trying to best categorize the customer query.
                               You are an agent for an AI products and hardware company.

                               Please read the customer query below and
                               determine the best category from the following list:

                               'Technical', 'Billing', or 'General'.

                               Remember:
                                - Technical queries will focus more on technical aspects like AI models, hardware, software related queries etc.
                                - General queries will focus more on general aspects like contacting support, finding things, policies etc.
                                - Billing queries will focus more on payment and purchase related aspects

                                Return just the category name (from one of the above)

                                Query:
                                {customer_query}
                            """
    prompt = ROUTE_CATEGORY_PROMPT.format(customer_query=query)
    route_category = llm.with_structured_output(QueryCategory).invoke(prompt)

    return {
        "query_category": route_category.categorized_topic
    }

In [15]:
def analyze_inquiry_sentiment(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Analyze the sentiment of the customer query as Positive, Neutral, or Negative.
    """

    query = support_state["customer_query"]
    SENTIMENT_CATEGORY_PROMPT = """Act as a customer support agent trying to best categorize the customer query's sentiment.
                                   You are an agent for an AI products and hardware company.

                                   Please read the customer query below,
                                   analyze its sentiment which should be one from the following list:

                                   'Positive', 'Neutral', or 'Negative'.

                                   Return just the sentiment (from one of the above)

                                   Query:
                                   {customer_query}
                                """
    prompt = SENTIMENT_CATEGORY_PROMPT.format(customer_query=query)
    sentiment_category = llm.with_structured_output(QuerySentiment).invoke(prompt)

    return {
        "query_sentiment": sentiment_category.sentiment
    }

In [16]:
analyze_inquiry_sentiment({"customer_query": "what is your refund policy?"})

{'query_sentiment': 'Neutral'}

In [17]:
analyze_inquiry_sentiment({"customer_query": "what is your refund policy? I am really fed up with this product and need to refund it"})

{'query_sentiment': 'Negative'}

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from typing import Dict

def generate_technical_response(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Provide a technical support response by combining knowledge from the vector store and LLM.
    """
    # Retrieve category and ensure it is lowercase for metadata filtering

    categorized_topic = support_state["query_category"]
    query = support_state["customer_query"]

    # Use metadata filter for 'technical' queries
    if categorized_topic.lower() == "technical":
        metadata_filter = {"category": "technical"}
        kbase_search.search_kwargs["filter"] = metadata_filter

        # Perform retrieval from VectorDB
        relevant_docs = kbase_search.invoke(query)
        retrieved_content = "\n\n".join(doc.page_content for doc in relevant_docs)

        # Combine retrieved information into the prompt
        prompt = ChatPromptTemplate.from_template(
            """
            Craft a clear and detailed technical support response for the following customer query.
            Use the provided knowledge base information to enrich your response.
            In case there is no knowledge base information or you do not know the answer just say:

            Apologies I was not able to answer your question, please reach out to +1-xxx-xxxx

            Customer Query:
            {customer_query}

            Relevant Knowledge Base Information:
            {retrieved_content}
            """
        )

        # Generate the final response using the LLM
        chain = prompt | llm
        tech_reply = chain.invoke({
            "customer_query": query,
            "retrieved_content": retrieved_content
        }).content
    else:
        # For non-technical queries, provide a default response or a general handling
        tech_reply = "Apologies I was not able to answer your question, please reach out to +1-xxx-xxxx"

    # Update and return the modified support state
    return {
        "final_response": tech_reply
    }


In [19]:
generate_technical_response({"customer_query": "what is your refund policy?", "query_category": "General"})

{'final_response': 'Apologies I was not able to answer your question, please reach out to +1-xxx-xxxx'}

In [20]:
response = generate_technical_response({"customer_query": "do you support on-prem models?", "query_category": "Technical"})
response

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{'final_response': 'Short answer: Yes — we support running our AI models on-premise and provide deployment guides to help you do that.\n\nWhat we provide\n- On-prem deployment: Our models can be deployed on your premises (physical servers or private cloud). We provide step-by-step deployment guides for multiple environments.\n- Pre-trained models: We offer a range of pre-trained models (NLP, computer vision, recommendation systems) you can download and run on-prem or use as a starting point for fine-tuning.\n- Troubleshooting guidance: If you experience performance or runtime issues, start with logs, resource utilization, and input-data validation; our support team can assist further.\n\nSuggested on-prem deployment checklist (typical next steps)\n1. Choose the model\n   - Pick a pre-trained model from our catalog or specify a custom model you want to deploy.\n2. Confirm environment & prerequisites\n   - OS, container runtime (Docker), or orchestration (Kubernetes) preferences\n   - Ha

In [21]:
from IPython.display import display, Markdown
display(Markdown(response["final_response"]))

Short answer: Yes — we support running our AI models on-premise and provide deployment guides to help you do that.

What we provide
- On-prem deployment: Our models can be deployed on your premises (physical servers or private cloud). We provide step-by-step deployment guides for multiple environments.
- Pre-trained models: We offer a range of pre-trained models (NLP, computer vision, recommendation systems) you can download and run on-prem or use as a starting point for fine-tuning.
- Troubleshooting guidance: If you experience performance or runtime issues, start with logs, resource utilization, and input-data validation; our support team can assist further.

Suggested on-prem deployment checklist (typical next steps)
1. Choose the model
   - Pick a pre-trained model from our catalog or specify a custom model you want to deploy.
2. Confirm environment & prerequisites
   - OS, container runtime (Docker), or orchestration (Kubernetes) preferences
   - Hardware: CPU-only or GPU (NVIDIA drivers, CUDA/cuDNN versions) and available memory
   - Network and security constraints (firewalls, private registries, authentication)
3. Select serving method
   - Single-machine REST server, containerized service, or scaled K8s deployment depending on expected load
4. Obtain artifacts & guides
   - Download the pre-trained model files and follow the matching deployment guide for your environment
5. Configure monitoring & logging
   - Enable logs and metrics so you can review inference latency, throughput, and resource usage
6. Validate and test
   - Run end-to-end tests with representative inputs and check outputs, performance, and stability

Basic troubleshooting steps (fast checklist)
- Review service and inference logs for errors or stack traces.
- Check CPU/GPU utilization, memory usage, and I/O to spot resource bottlenecks.
- Validate input data format, tokenization, and preprocessing steps match model expectations.
- Test a minimal local deployment to isolate environment issues from model issues.

How I can help next
- Tell me:
  - Which model or use case (NLP, vision, recommender)?
  - Your target environment (OS, Docker/Kubernetes, bare metal)?
  - Available hardware (GPU type/driver versions, or CPU-only) and expected throughput/latency targets?
I will then provide the specific on-prem deployment guide and checklist tailored to your environment, or escalate to our support team if you prefer hands-on assistance.

In [22]:
def generate_billing_response(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Provide a billing support response by combining knowledge from the vector store and LLM.
    """
    # Retrieve category and ensure it is lowercase for metadata filtering
    categorized_topic = support_state["query_category"]
    query = support_state["customer_query"]

    # Use metadata filter for 'billing' queries
    if categorized_topic.lower() == "billing":
        metadata_filter = {"category": "billing"}
        kbase_search.search_kwargs["filter"] = metadata_filter

        # Perform retrieval from VectorDB
        relevant_docs = kbase_search.invoke(query)
        retrieved_content = "\n\n".join(doc.page_content for doc in relevant_docs)

        # Combine retrieved information into the prompt
        prompt = ChatPromptTemplate.from_template(
            """
            Craft a clear and detailed billing support response for the following customer query.
            Use the provided knowledge base information to enrich your response.
            In case there is no knowledge base information or you do not know the answer just say:

            Apologies I was not able to answer your question, please reach out to +1-xxx-xxxx

            Customer Query:
            {customer_query}

            Relevant Knowledge Base Information:
            {retrieved_content}
            """
        )

        # Generate the final response using the LLM
        chain = prompt | llm
        billing_reply = chain.invoke({
            "customer_query": query,
            "retrieved_content": retrieved_content
        }).content
    else:
        # For non-billing queries, provide a default response or a general handling
        billing_reply = "Apologies I was not able to answer your question, please reach out to +1-xxx-xxxx"

    # Update and return the modified support state
    return {
        "final_response": billing_reply
    }

In [23]:
generate_billing_response({"customer_query": "what payment methods are supported?", "query_category": "Billing"})

{'final_response': 'Thanks — here are the payment methods we accept:\n\n- Credit cards (for standard accounts)  \n- PayPal  \n- Wire transfers (available for corporate accounts)\n\nA few related billing notes:\n- Applicable taxes are calculated based on your location and are displayed during checkout.  \n- There are no hidden fees; however, optional add-ons (for example, premium support) may incur extra charges.\n\nIf you’d like more details, tell me:\n- which payment method you plan to use (I can share next steps), or  \n- whether you’re a corporate account (I can provide wire transfer instructions).'}

In [24]:
def generate_general_response(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Provide a general support response by combining knowledge from the vector store and LLM.
    """
    # Retrieve category and ensure it is lowercase for metadata filtering
    categorized_topic = support_state["query_category"]
    query = support_state["customer_query"]

    # Use metadata filter for 'general' queries
    if categorized_topic.lower() == "general":
        metadata_filter = {"category": "general"}
        kbase_search.search_kwargs["filter"] = metadata_filter

        # Perform retrieval from VectorDB
        relevant_docs = kbase_search.invoke(query)
        retrieved_content = "\n\n".join(doc.page_content for doc in relevant_docs)

        # Combine retrieved information into the prompt
        prompt = ChatPromptTemplate.from_template(
            """
            Craft a clear and detailed general support response for the following customer query.
            Use the provided knowledge base information to enrich your response.
            In case there is no knowledge base information or you do not know the answer just say:

            Apologies I was not able to answer your question, please reach out to +1-xxx-xxxx

            Customer Query:
            {customer_query}

            Relevant Knowledge Base Information:
            {retrieved_content}
            """
        )

        # Generate the final response using the LLM
        chain = prompt | llm
        general_reply = chain.invoke({
            "customer_query": query,
            "retrieved_content": retrieved_content
        }).content
    else:
        # For non-general queries, provide a default response or a general handling
        general_reply = "Apologies I was not able to answer your question, please reach out to +1-xxx-xxxx"

    # Update and return the modified support state
    return {
        "final_response": general_reply
    }


In [25]:
generate_general_response({"customer_query": "what is your refund policy?", "query_category": "General"})

{'final_response': 'Thanks for asking — here’s our refund policy and how to start a return:\n\n- Refund policy: We offer a 30-day money‑back guarantee for all our products.  \n- How to initiate: Please contact our support team to start a refund. When you reach out, have your order number and purchase date available; photos or a brief description of the reason for the return (if applicable) will help speed things up. Our support team will walk you through the next steps.\n\nSpecial note for damaged hardware:\n- If you receive damaged hardware, please report it to support within 48 hours. We will arrange a replacement; if you would prefer a refund instead, mention that when you report the damage and support will advise on options.\n\nIf you’d like, share your order number here (or contact support directly) and I can help get the process started.'}

In [26]:
def escalate_to_human_agent(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Escalate the query to a human agent if sentiment is negative.
    """

    return {
        "final_response": "Apologies, we are really sorry! Someone from our team will be reaching out to your shortly!"
    }

In [27]:
def determine_route(support_state: CustomerSupportState) -> str:
    """
    Route the inquiry based on sentiment and category.
    """
    if support_state["query_sentiment"] == "Negative":
        return "escalate_to_human_agent"
    elif support_state["query_category"] == "Technical":
        return "generate_technical_response"
    elif support_state["query_category"] == "Billing":
        return "generate_billing_response"
    else:
        return "generate_general_response"

### 5.3 Compile the LangGraph Workflow

Wire the nodes together — `categorize_inquiry → analyze_inquiry_sentiment → (conditional route) → one of the four reply nodes → END`. Compile with a `MemorySaver` checkpointer so each `thread_id` carries its own conversation state.

In [28]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# Create the graph with our typed state
customer_support_graph = StateGraph(CustomerSupportState)

# Add nodes for each function
customer_support_graph.add_node("categorize_inquiry", categorize_inquiry)
customer_support_graph.add_node("analyze_inquiry_sentiment", analyze_inquiry_sentiment)
customer_support_graph.add_node("generate_technical_response", generate_technical_response)
customer_support_graph.add_node("generate_billing_response", generate_billing_response)
customer_support_graph.add_node("generate_general_response", generate_general_response)
customer_support_graph.add_node("escalate_to_human_agent", escalate_to_human_agent)

# Add edges to represent the processing flow
customer_support_graph.add_edge("categorize_inquiry", "analyze_inquiry_sentiment")
customer_support_graph.add_conditional_edges(
    "analyze_inquiry_sentiment",
    determine_route,
    [
        "generate_technical_response",
        "generate_billing_response",
        "generate_general_response",
        "escalate_to_human_agent"
    ]
)

# All terminal nodes lead to the END
customer_support_graph.add_edge("generate_technical_response", END)
customer_support_graph.add_edge("generate_billing_response", END)
customer_support_graph.add_edge("generate_general_response", END)
customer_support_graph.add_edge("escalate_to_human_agent", END)

# Set the entry point for the workflow
customer_support_graph.set_entry_point("categorize_inquiry")

# Compile the graph into a runnable agent
memory = MemorySaver()
compiled_support_agent = customer_support_graph.compile(checkpointer=memory)

### 5.4 Visualise the Workflow

A Mermaid PNG of the compiled graph. If the renderer is unavailable in your venv (no `pygraphviz`), we fall back to ASCII.

In [ ]:
from IPython.display import display, Image, Markdown

try:
    display(Image(compiled_support_agent.get_graph().draw_mermaid_png()))
except Exception as exc:
    print('Mermaid PNG renderer not available — printing the ASCII graph instead.')
    print(compiled_support_agent.get_graph().draw_ascii())
    print(f'(reason: {exc})')


## 6. Smoke Test the Agent (with LangSmith Tracing Live)

Every `agent.stream(...)` invocation below now appears as a **trace** in your LangSmith project — open https://smith.langchain.com → project `eclerx-day9-lab9.1-customer-support-router` after running these cells to inspect the spans, token counts, and latencies.

### 6.1 Helper to Run the Workflow

Streams events back from the compiled agent and prints the final response. Each invocation is one LangSmith trace.

In [31]:
def call_support_agent(agent, prompt, user_session_id, verbose=False):
    events = agent.stream(
        {"customer_query": prompt}, # initial state of the agent
        {"configurable": {"thread_id": user_session_id}},
        stream_mode="values",
    )

    print('Running Agent. Please wait...')
    for event in events:
        if verbose:
                print(event)

    display(Markdown(event['final_response']))

### 6.2 Smoke Tests

Three queries — one Technical, one Billing, one General — to verify routing + RAG + sentiment paths. After running, open the LangSmith dashboard for the project and confirm three traces appear.

In [32]:
[item['text'] for item in knowledge_base]

['Question: How do I integrate your AI product with my existing CRM system? Answer: You can integrate our AI product with your CRM using our API. Refer to the API documentation available on our website for step-by-step guidance.',
 'Question: What programming languages are supported by your SDK? Answer: Our SDK supports Python, Java, and JavaScript. Additional language support is planned for future updates.',
 'Question: Can your AI models run on-premise? Answer: Yes, our AI models can be deployed on-premise. We provide deployment guides for various environments.',
 'Question: Does your hardware support edge AI applications? Answer: Yes, our hardware is optimized for edge AI, with low-latency processing and energy-efficient designs.',
 'Question: How do I troubleshoot issues with model performance? Answer: Start by reviewing the logs, checking resource utilization, and validating input data quality. You can also reach out to support for assistance.',
 'Question: Can I fine-tune your AI

In [33]:
uid = 'prash001'
query = "do you support pre-trained models?"
call_support_agent(agent=compiled_support_agent,
                   prompt=query,
                   user_session_id=uid,
                   verbose=True)

Running Agent. Please wait...
{'customer_query': 'do you support pre-trained models?'}
{'customer_query': 'do you support pre-trained models?', 'query_category': 'Technical'}
{'customer_query': 'do you support pre-trained models?', 'query_category': 'Technical', 'query_sentiment': 'Neutral'}
{'customer_query': 'do you support pre-trained models?', 'query_category': 'Technical', 'query_sentiment': 'Neutral', 'final_response': 'Yes — we do support pre-trained models.\n\nSummary of what’s available\n- Types: We offer a range of pre-trained models for common tasks including NLP (text classification, generation, NER, etc.), computer vision (image classification, detection, segmentation), and recommendation systems.\n- Fine‑tuning: You can fine-tune our models on your own datasets through the platform.\n- Deployment: Models can be deployed on-premise; we provide deployment guides for multiple environments.\n\nTypical workflow (high-level)\n1. Choose a model: Browse the model catalog in the d

Yes — we do support pre-trained models.

Summary of what’s available
- Types: We offer a range of pre-trained models for common tasks including NLP (text classification, generation, NER, etc.), computer vision (image classification, detection, segmentation), and recommendation systems.
- Fine‑tuning: You can fine-tune our models on your own datasets through the platform.
- Deployment: Models can be deployed on-premise; we provide deployment guides for multiple environments.

Typical workflow (high-level)
1. Choose a model: Browse the model catalog in the dashboard/portal and pick a pre-trained model that fits your task.
2. Evaluate: Run inference on a small sample of your data to verify baseline performance.
3. Fine-tune (optional): If you need higher accuracy or domain adaptation, upload your labeled dataset and start a fine-tuning job using the platform’s fine-tuning workflow (configure hyperparameters, monitor training, validate).
4. Deploy: Deploy the resulting model either to our managed environment or on-premise using the provided deployment guides. The guides cover common environments and help with packaging, resource requirements, and serving endpoints.

How I can help next
- Tell me your use case (task type), whether you want to fine-tune, and your target environment (cloud or on‑prem). Also share framework preference (PyTorch/TensorFlow) and dataset size if known.
- I can then: recommend specific pre-trained models, outline expected compute and storage needs, and walk you step-by-step through fine-tuning or on‑prem deployment.

If you want immediate instructions for a specific task, provide the details above and I’ll give targeted next steps.

In [34]:
query = "how do I get my invoice?"
call_support_agent(agent=compiled_support_agent,
                   prompt=query,
                   user_session_id=uid,
                   verbose=True)

Running Agent. Please wait...
{'customer_query': 'how do I get my invoice?', 'query_category': 'Technical', 'query_sentiment': 'Neutral', 'final_response': 'Yes — we do support pre-trained models.\n\nSummary of what’s available\n- Types: We offer a range of pre-trained models for common tasks including NLP (text classification, generation, NER, etc.), computer vision (image classification, detection, segmentation), and recommendation systems.\n- Fine‑tuning: You can fine-tune our models on your own datasets through the platform.\n- Deployment: Models can be deployed on-premise; we provide deployment guides for multiple environments.\n\nTypical workflow (high-level)\n1. Choose a model: Browse the model catalog in the dashboard/portal and pick a pre-trained model that fits your task.\n2. Evaluate: Run inference on a small sample of your data to verify baseline performance.\n3. Fine-tune (optional): If you need higher accuracy or domain adaptation, upload your labeled dataset and start a 

Hi — you can download your invoice from your account dashboard. Here’s how:

1. Sign in to your account.  
2. Open your Account Dashboard and go to the "Billing" (or "Billing & Payments"/"Invoices") section.  
3. Find the invoice for the purchase (by date or order number) and click View or Download — you can usually download a PDF or print it directly.

If you don’t see the invoice:
- Confirm you’re signed into the account used to make the purchase.  
- Check that the purchase completed (and allow up to 24 hours for processing).  
- Check your email (including spam) for a receipt.  
- If the billing email on file needs updating, you can update it from the same Billing section in your account dashboard.

Need a consolidated bill for multiple accounts? That’s available for enterprise customers — please contact Sales to enable consolidated billing.

If the invoice still isn’t available or you’d like me to help locate a specific invoice, reply with your account email, order number, and purchase date (do not share payment card details) and I’ll assist.

In [35]:
query = "Can you tell me about your shipping policy?"
call_support_agent(agent=compiled_support_agent,
                   prompt=query,
                   user_session_id=uid,
                   verbose=False)

Running Agent. Please wait...


Thanks — here’s a summary of our shipping policy and related customer-care policies:

Shipping (hardware)
- Free standard shipping for orders over $500.
- For orders under $500, a flat shipping fee of $20 applies.
- Standard transit time is typically 5–7 business days.

Damaged deliveries
- If you receive damaged hardware, please report it to our support team within 48 hours of delivery. We will arrange a replacement.

Refunds
- We offer a 30-day money‑back guarantee on all products. Please contact support to initiate a refund.

If you need anything specific (status of an order, tracking number, expedited or international shipping options, or to report damage), please reply with your order number and a brief description or contact our support team so we can assist you directly.

## 7. Custom Callbacks for Audit Logging

LangSmith captures everything we need for **debugging** and **evaluation**. But for a regulated bank or financial-services client, the audit team also wants:

- A **tamper-evident** local log (JSON-lines, append-only, can be shipped to Splunk / SIEM).
- Logs that **survive even if LangSmith is unreachable** or the trace upload fails.
- A guarantee that **PII is redacted before any external system sees it** (we redact in the callback,   before LangSmith ever receives the data — see Lab 9.4 for the redaction pipeline).

The right primitive is a custom **`BaseCallbackHandler`**. It runs in-process, on every LLM call, tool call, chain start/end, and error — synchronously — and we control exactly what it persists.

*This is the M3 → M9 reassignment from the curriculum spec — the audit-callback pattern lives here, not in the LangChain Deep-Dive module, because audit & compliance is genuinely a Day-9 topic.*

### 7.1 Subclass `BaseCallbackHandler`

We override five lifecycle hooks: `on_llm_start`, `on_llm_end`, `on_tool_start`, `on_tool_end`, `on_chain_error`. Each hook captures (timestamp, run_id, parent_run_id, node_name, tokens, latency, error) and writes one JSON object per line to `audit.jsonl`. The schema is simple enough to grep, rich enough for forensics.

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from uuid import UUID

from langchain_core.callbacks import BaseCallbackHandler

AUDIT_LOG_PATH = Path('./audit.jsonl')


class JsonLinesAuditHandler(BaseCallbackHandler):
    """Append-only audit log of every LLM call, tool call, and chain error.

    Each event is written as a single JSON object on its own line — Splunk-, jq-, and
    pandas-friendly. Designed for regulated environments where LangSmith Cloud is
    not approved for outbound data.
    """

    def __init__(self, log_path: Path = AUDIT_LOG_PATH) -> None:
        self.log_path = log_path
        self._llm_starts: dict[UUID, float] = {}
        self._tool_starts: dict[UUID, float] = {}

    def _emit(self, event: dict[str, Any]) -> None:
        event['ts'] = datetime.now(timezone.utc).isoformat()
        with self.log_path.open('a', encoding='utf-8') as f:
            f.write(json.dumps(event, default=str) + '\n')

    def on_llm_start(self, serialized, prompts, *, run_id, parent_run_id=None, **kwargs):
        self._llm_starts[run_id] = time.perf_counter()
        self._emit({
            'event': 'llm_start',
            'run_id': str(run_id),
            'parent_run_id': str(parent_run_id) if parent_run_id else None,
            'model': (serialized or {}).get('id', ['unknown'])[-1],
            'prompt_chars': sum(len(p) for p in prompts),
        })

    def on_llm_end(self, response, *, run_id, parent_run_id=None, **kwargs):
        latency_ms = (time.perf_counter() - self._llm_starts.pop(run_id, time.perf_counter())) * 1000
        usage = (response.llm_output or {}).get('token_usage', {}) if response.llm_output else {}
        self._emit({
            'event': 'llm_end',
            'run_id': str(run_id),
            'parent_run_id': str(parent_run_id) if parent_run_id else None,
            'latency_ms': round(latency_ms, 1),
            'prompt_tokens': usage.get('prompt_tokens'),
            'completion_tokens': usage.get('completion_tokens'),
            'total_tokens': usage.get('total_tokens'),
        })

    def on_tool_start(self, serialized, input_str, *, run_id, parent_run_id=None, **kwargs):
        self._tool_starts[run_id] = time.perf_counter()
        self._emit({
            'event': 'tool_start',
            'run_id': str(run_id),
            'parent_run_id': str(parent_run_id) if parent_run_id else None,
            'tool': (serialized or {}).get('name'),
            'input_chars': len(input_str),
        })

    def on_tool_end(self, output, *, run_id, parent_run_id=None, **kwargs):
        latency_ms = (time.perf_counter() - self._tool_starts.pop(run_id, time.perf_counter())) * 1000
        self._emit({
            'event': 'tool_end',
            'run_id': str(run_id),
            'parent_run_id': str(parent_run_id) if parent_run_id else None,
            'latency_ms': round(latency_ms, 1),
            'output_chars': len(str(output)),
        })

    def on_chain_error(self, error, *, run_id, parent_run_id=None, **kwargs):
        self._emit({
            'event': 'chain_error',
            'run_id': str(run_id),
            'parent_run_id': str(parent_run_id) if parent_run_id else None,
            'error_type': type(error).__name__,
            'error_msg': str(error)[:500],
        })


audit_handler = JsonLinesAuditHandler()
print(f'Audit handler ready. Log path: {AUDIT_LOG_PATH.resolve()}')


### 7.2 Re-run the Agent with the Audit Callback Attached

Pass the handler via `config={'callbacks': [audit_handler]}`. LangSmith continues to trace as well — the two are independent. The callback runs *in-process* so even if the LangSmith upload fails, the JSON-lines log is durable on disk.

In [ ]:
# Reset the audit log for a clean comparison
if AUDIT_LOG_PATH.exists():
    AUDIT_LOG_PATH.unlink()

audit_queries = [
    ('audit-001', 'Do you support pre-trained vision models?'),
    ('audit-002', 'How do I download my last invoice?'),
    ('audit-003', 'I am furious — your hardware bricked itself overnight, refund NOW.'),
]

for session_id, query in audit_queries:
    print(f'\n── {session_id} · {query[:60]}…')
    events = compiled_support_agent.stream(
        {'customer_query': query},
        config={
            'configurable': {'thread_id': session_id},
            'callbacks': [audit_handler],   # ← audit log + LangSmith both fire
            'metadata': {'lab': '9.1', 'session_id': session_id},
        },
        stream_mode='values',
    )
    final = None
    for ev in events:
        final = ev
    print(f'   sentiment={final.get("query_sentiment")} category={final.get("query_category")}')


### 7.3 Inspect the Audit Log

Load the JSON-lines file and walk through the recorded events. In production this is what your SOC team grep / Splunks at 3am when an incident lands.

In [ ]:
import json
from collections import Counter

with AUDIT_LOG_PATH.open(encoding='utf-8') as f:
    events = [json.loads(line) for line in f if line.strip()]

print(f'Total events captured: {len(events)}')
print('Event types:', Counter(e['event'] for e in events))

print('\nFirst 5 events:')
for ev in events[:5]:
    print(json.dumps(ev, indent=2))


In [ ]:
# Quick aggregation: total tokens by event vs total wall-clock latency by event.
# This is the kind of report a FinOps team produces from the audit log.
import pandas as pd

df = pd.DataFrame(events)
if 'total_tokens' in df.columns:
    print('Total LLM tokens consumed:', int(df['total_tokens'].dropna().sum()))
if 'latency_ms' in df.columns:
    print('Avg LLM latency (ms):     ', round(df.loc[df['event'] == 'llm_end', 'latency_ms'].mean(), 1))
df.head(10)


### 7.4 Regulatory Implications — When You Need This

| Requirement | Why a custom callback (not just LangSmith) | Examples |
|---|---|---|
| **Data residency** | Some EU/IN/UAE clients cannot ship traces to a US-region SaaS | EU GDPR Art 44 transfers, RBI data-localisation, UAE CBUAE Sandbox |
| **Tamper-evident audit trail** | Append-only file → SIEM is auditor's preferred chain-of-custody | OCC 2011-12 model-risk SR 11-7 |
| **PII redaction before egress** | Callback fires *before* trace upload — strip secrets in-process | EU GDPR, CPRA, India DPDP Act |
| **Custom domain metrics** | LangSmith captures generic spans; you may need a *credit-decision* event | Internal MRM frameworks |
| **Survivability when SaaS is down** | Local file persists even if LangSmith Cloud is unreachable | Ops resilience requirements |

Production rule of thumb: **always run both** (LangSmith for human debugging + custom callback for the auditor's chain-of-custody). They are complementary, not alternatives.

## 8. LangSmith Datasets and Evaluators

Datasets, evaluators, and experiments turn LangSmith from a *debugger* into a *regression-test framework*. Every prompt change, every model swap, every routing-rule tweak — re-run the experiment and compare scores.

**Vocabulary.** *Dataset* = a versioned set of (input, expected-output) examples. *Evaluator* = a function that scores one (input, expected, actual) tuple. *Experiment* = one full run of a dataset through your system, with all evaluators attached, producing a comparison view in the dashboard.

Below we (i) build a small routing dataset, (ii) write a heuristic correctness evaluator, (iii) kick off an experiment. The evaluator is intentionally simple — Lab 9.3 covers the LLM-as-Judge pattern that production agents rely on.

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

client = Client()
DATASET_NAME = 'eclerx-day9-lab9.1-routing-eval'

examples = [
    {'inputs': {'customer_query': 'Do you ship to Singapore?'},        'outputs': {'expected_category': 'General'}},
    {'inputs': {'customer_query': 'My GPU appliance is throwing CUDA errors after the latest firmware'},
        'outputs': {'expected_category': 'Technical'}},
    {'inputs': {'customer_query': 'Why was my Visa charged twice this month?'},
        'outputs': {'expected_category': 'Billing'}},
    {'inputs': {'customer_query': 'How do I update my saved payment method?'},
        'outputs': {'expected_category': 'Billing'}},
    {'inputs': {'customer_query': 'I want a refund right now, your product is unusable'},
        'outputs': {'expected_category': 'Billing'}},  # routing should still hit Billing; sentiment is what triggers escalation
    {'inputs': {'customer_query': 'Does your SDK support Python 3.12?'},
        'outputs': {'expected_category': 'Technical'}},
    {'inputs': {'customer_query': 'What are your weekend support hours?'},
        'outputs': {'expected_category': 'General'}},
]

# Idempotent dataset creation: only create if it does not exist already
try:
    ds = client.read_dataset(dataset_name=DATASET_NAME)
    print(f'Dataset already exists: {ds.id}')
except Exception:
    ds = client.create_dataset(
        dataset_name=DATASET_NAME,
        description='Routing-accuracy benchmark for the customer-support router agent.',
    )
    client.create_examples(
        inputs=[e['inputs'] for e in examples],
        outputs=[e['outputs'] for e in examples],
        dataset_id=ds.id,
    )
    print(f'Created dataset {ds.id} with {len(examples)} examples.')


In [ ]:
# Define the target — a thin wrapper around our existing categorize_inquiry node so the
# dataset's inputs flow straight in and we surface only the predicted category.
def routing_target(inputs: dict) -> dict:
    state = categorize_inquiry({'customer_query': inputs['customer_query']})
    return {'predicted_category': state['query_category']}


# Define a heuristic evaluator. In production you'd register multiple evaluators —
# correctness, latency, cost, faithfulness — see Lab 9.3.
def correctness_evaluator(run, example) -> dict:
    pred = (run.outputs or {}).get('predicted_category')
    expected = (example.outputs or {}).get('expected_category')
    score = 1.0 if pred == expected else 0.0
    return {'key': 'routing_correctness', 'score': score, 'comment': f'pred={pred} expected={expected}'}


In [ ]:
# Run the experiment. Results land in the LangSmith dashboard under the dataset's Experiments tab.
results = evaluate(
    routing_target,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix='lab9.1-baseline',
    metadata={'lab': '9.1', 'agent': 'router-v1'},
)

print('Experiment complete. Inspect at the URL above (LangSmith dashboard).')
print('Pattern: change a prompt → re-run this cell with a new experiment_prefix → compare diff side-by-side.')


## 9. LangSmith Hub for Prompt Versioning

The Hub is **git for prompts**. Push a prompt with a name and an optional tag; pull it by name (and optionally by version). Once your prompts live in the Hub, your CI/CD pipeline can:

1. Detect a Hub change → kick off the dataset experiment from §8 → block the deploy if scores regress.
2. Pin a *production* tag separate from *staging* — `hub.pull('routing-prompt:production')`.
3. Roll back instantly without redeploying code — just re-tag.

Below we push the routing prompt currently inlined in `categorize_inquiry`, then pull it back as a verifiable round-trip.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client

ROUTING_PROMPT_NAME = 'eclerx-day9-router-categorization'

routing_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a customer support agent for an AI products and hardware company. '
               'Classify the customer query into exactly one of: Technical, Billing, General. '
               'Return only the category name.'),
    ('human',  '{customer_query}'),
])

client = Client()

try:
    pushed_url = client.push_prompt(ROUTING_PROMPT_NAME, object=routing_prompt)
    print(f'Pushed prompt → {pushed_url}')
except Exception as exc:
    # push_prompt requires write permissions on the workspace — common to fail on free-tier or
    # read-only API keys. The pull below still works against a public prompt.
    print(f'Push failed (permissions?): {exc}')


In [ ]:
# Pull the prompt back by name. In CI you would pin a tag, e.g. f'{ROUTING_PROMPT_NAME}:production'.
from langchain import hub

try:
    pulled_prompt = hub.pull(ROUTING_PROMPT_NAME)
    print('Pulled prompt — type:', type(pulled_prompt).__name__)
    print('Round-tripped messages:')
    for msg in pulled_prompt.messages:
        print(' ', type(msg).__name__, '→', str(msg.prompt.template)[:120])
except Exception as exc:
    print(f'Pull failed: {exc}')
    print('If push succeeded above, retry after a few seconds (Hub is eventually-consistent).')


### 9.1 CI/CD Workflow with the Hub

```
  developer edits prompt locally
         │
         ▼
  client.push_prompt('routing-prompt', tag='staging')
         │
         ▼
  CI job: evaluate(target, dataset=routing-eval) ──→ score regression?
         │                                              │
         ▼                                              ▼
  re-tag :production                              block PR / open issue
         │
         ▼
  prod app: hub.pull('routing-prompt:production')
```

Key insight: **prompts ship through the same gate as code**. No more silent prompt edits in production — every change has a Hub commit, an experiment run, and an audit trail.

---
## 10. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **LangSmith auto-tracing** | `LANGSMITH_TRACING=true` + a project name is *all* you need — every LangChain / LangGraph call is captured. |
| **Custom `BaseCallbackHandler`** | Override the lifecycle hooks to write a tamper-evident JSON-lines audit log that survives even when LangSmith Cloud is unreachable. |
| **Datasets / Evaluators / Experiments** | Datasets = versioned (input, expected) sets · Evaluators = scoring functions · Experiments = the full regression run. |
| **LangSmith Hub** | Git for prompts. Tag :production / :staging. Lets your CI/CD pipeline gate prompt changes the same way it gates code. |
| **LangSmith vs custom callback** | Run BOTH in regulated environments. LangSmith for debugging; custom callback for the auditor's chain-of-custody. |

**Next Lab:** Lab 9.2 — Langfuse: Open-Source Observability (when LangSmith Cloud is off the table) 📊


## 11. Stretch Exercise (Optional)

1. Add an `on_llm_error` hook to `JsonLinesAuditHandler` and force a failure (e.g. invalid model id) to verify the error path writes a row.
2. Extend the dataset in §8 to 30 examples, add a `latency_evaluator`, and run an experiment that compares `gpt-5-mini` vs `claude-haiku-4-5` as the routing LLM.
3. Wrap the JSON-lines write in `RotatingFileHandler` semantics so the audit file rolls at 100 MB — production-readiness fix.
4. Add a `metadata={'session_id': ..., 'tenant_id': ...}` field to every `agent.stream(config=...)` call. Confirm both LangSmith spans AND the audit log carry the metadata.
5. Push two versions of the routing prompt to the Hub (e.g. terse vs chain-of-thought). Use the experiment from §8 to compare and pick the winner; tag winner as `:production`.


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. What does LangSmith automatically trace, and what do you have to add manually?**

*Hint:* Auto-instrumentation vs custom spans.

*Answer sketch:* LangSmith auto-traces every LangChain Runnable, every LangGraph node, every LLM call, every tool call, and every retriever — when `LANGSMITH_TRACING=true`. You add manually: any plain-Python function not wrapped in a Runnable (use the `@traceable` decorator), custom metadata/tags, custom feedback scores via `client.create_feedback`, and any external HTTP/DB calls you want spans for.

---

**Q2. When use a custom `BaseCallbackHandler` vs LangSmith's built-in tracing?**

*Hint:* On-prem audit needs vs SaaS debugging.

*Answer sketch:* LangSmith is for *human debugging* and *eval workflows* — generic spans, latencies, replay. A custom `BaseCallbackHandler` is for (a) on-prem / regulated environments where outbound traces are blocked, (b) tamper-evident audit logs your SOC ships to a SIEM, (c) PII redaction *before* egress to any SaaS, and (d) custom-domain metrics LangSmith does not capture (e.g., a credit-decision event). Production pattern: run **both**.

---

**Q3. What's the difference between a LangSmith dataset, an evaluator, and an experiment?**

*Hint:* Three nouns, three roles.

*Answer sketch:* **Dataset** = a versioned set of (input, expected-output) examples — your test fixture. **Evaluator** = a function that scores one (input, expected, actual) triple — heuristic, LLM-as-Judge, or human. **Experiment** = one full run of a dataset through a target system with all evaluators attached, producing a comparison view in the dashboard. Repeat experiments after every change → regression matrix.

---

**Q4. How does LangSmith Hub help in a CI/CD pipeline?**

*Hint:* Git-for-prompts, decoupled deploys.

*Answer sketch:* Hub stores every prompt as a versioned, taggable artifact. CI watches Hub changes, automatically kicks off the dataset experiment, blocks the merge if scores regress, and only re-tags `:production` once approved. Apps in prod do `hub.pull('prompt:production')` so a rollback is just a re-tag — no redeploy. Result: prompt changes flow through the same gates as code changes.

---

**Q5. What kind of data does a custom callback typically capture for audit?**

*Hint:* What would an auditor demand at 3am during an incident?

*Answer sketch:* Per event: ISO-8601 timestamp, run_id + parent_run_id (for trace stitching), node/tool name, model id, prompt token count + completion token count + total cost, latency in ms, error type + truncated message. Per session: session_id, tenant_id, user_id (post-redaction), correlation IDs into upstream systems. Append-only JSON-lines on disk → SIEM forwarder → 7-year retention bucket.

---

**Q6. How do you handle PII in traces — what are the controls?**

*Hint:* Redact at egress, never at ingest.

*Answer sketch:* Three layers: (1) before invocation — sanitise inputs in your app; (2) in the callback — pre-egress regex/Comprehend redaction (Lab 9.4) inside `on_llm_start` so the SaaS never sees the raw PII; (3) at LangSmith config — set `LANGSMITH_HIDE_INPUTS=true` / `LANGSMITH_HIDE_OUTPUTS=true` so payloads are never sent at all (only metadata + token counts). For the most-regulated tenants, skip LangSmith Cloud entirely and self-host LangSmith or switch to Langfuse self-hosted (Lab 9.2).

---

**Q7. When should you NOT send traces to LangSmith Cloud (regulated environments)?**

*Hint:* Data-residency, sovereignty, and contractual reasons.

*Answer sketch:* (a) **Customer PII / PHI** crossing borders that GDPR / HIPAA / DPDP / RBI rules forbid. (b) **Trade secrets** in prompts you cannot disclose to a third party. (c) **Government / defence** workloads under FedRAMP-High or equivalent. (d) **Bank model-risk policy** that requires single-tenant hosting. In these cases: self-host LangSmith on the customer's VPC, switch to Langfuse self-hosted (Lab 9.2), or rely entirely on the custom-callback audit log + a private OpenTelemetry collector.

---

**Q8. How do you run a regression eval automatically on prompt changes?**

*Hint:* CI hook on Hub commit → experiment → block-on-regression.

*Answer sketch:* 1) Authoritative dataset lives in LangSmith (versioned). 2) Authoritative prompts live in the Hub (versioned). 3) GitHub Actions / GitLab CI watches Hub commits (webhook). 4) On change, the runner calls `evaluate(target, data='routing-eval', evaluators=[...])` and parses the score. 5) If score on any evaluator regresses by more than a threshold (e.g. 5 %), the runner posts a failing status to the PR / blocks the Hub `:production` re-tag. 6) On pass, it re-tags and the prod app's next `hub.pull(':production')` picks up the new version.

